## Fine-Tuning de LLM no Azure AI Foundry

Neste notebook realizamos o fine-tuning de um modelo LLM utilizando o dataset preparado no notebook anterior.

### Objetivo

Treinar um modelo capaz de responder perguntas médicas com base em evidências científicas.

### Etapas

1. Conectar ao Azure OpenAI
2. Fazer upload do dataset de treinamento
3. Criar o job de fine-tuning
4. Monitorar o treinamento
5. Utilizar o modelo treinado

### Importar bibliotecas

In [3]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import time
import os
from IPython.display import clear_output

### Upload do dataset

O dataset é enviado para o Azure para ser utilizado no treinamento.

Usaremos o arquivo gerado no notebook anterior: B:\Pós\hospital-ai-diagnosis\dados\fine_tuning\llm_ready\pubmedqa_finetune.jsonl

In [ ]:
load_dotenv()


client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_API_VERSION")
)

training_file = client.files.create(
    file=open("B:/Pós/hospital-ai-diagnosis/dados/fine_tuning/llm_ready/pubmedqa_finetune.jsonl", "rb"),
    purpose="fine-tune"
)

training_file


### Monitorar se o arquivo foi processado

In [6]:
while True:
    
    file_status = client.files.retrieve(training_file.id)
    
    print(file_status.status)

    if file_status.status == "processed":
        print("Dataset pronto para treinamento")
        break

    time.sleep(5)

NameError: name 'training_file' is not defined

### Criar o job de fine-tuning

Aqui iniciamos o treinamento do modelo utilizando o dataset enviado.

O Azure cria um job que executa o fine-tuning do modelo base.

### Modelo base

O fine-tuning é realizado sobre um modelo base disponibilizado pelo Azure OpenAI.

Esse modelo já possui conhecimento geral e o treinamento adicional permite especializá-lo para responder perguntas médicas.

### Por que utilizar Fine-Tuning

O fine-tuning permite adaptar o comportamento de um modelo de linguagem para um domínio específico.

Neste projeto o objetivo é melhorar a qualidade das respostas relacionadas a diagnósticos médicos.

In [7]:
model = os.getenv("AZURE_MODEL_NAME")

fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    model=model
)

fine_tune_job

NameError: name 'training_file' is not defined

### Monitoramento
Esta chamada permite verificar o status do treinamento e capturar o modelo.  
Uma vez que o status do job for **succeded** o modelo esta pronto para teste.  

- Log do tempo de espera na tela para verificar o tempo de treinamento com o dataset utilizado.
- Também é possível monitorar no painel de fine-tuning do Azure Foundry



In [11]:
start = time.time()

while True:

    jobs = client.fine_tuning.jobs.list(limit=1)
    job = jobs.data[0]

    clear_output(wait=True)

    elapsed = int(time.time() - start)

    print("Job:", job.id)
    print("Status:", job.status)
    print("Tempo aguardando:", elapsed, "segundos")

    if job.status == "succeeded":
        print("Modelo pronto")
        print("Modelo:", job.fine_tuned_model)
        break

    time.sleep(30)


Job: ftjob-6798ea67892a4ca98210871b9d61757f
Status: succeeded
Tempo aguardando: 0 segundos
Modelo pronto
Modelo: gpt-4o-mini-2024-07-18.ft-6798ea67892a4ca98210871b9d61757f


### Teste do modelo treinado
Testamos o modelo treinado realizando uma pergunta clínica.

In [13]:
response = client.chat.completions.create(
    model= os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    messages=[
        {"role": "system", "content": "You are a medical assistant."},
        {"role": "user", "content": "Do mitochondria play a role in programmed cell death?"}
    ]
)

print(response.choices[0].message.content)

Yes, mitochondria play a significant role in programmed cell death, also known as apoptosis. They are often referred to as the "powerhouses" of the cell because they produce energy in the form of adenosine triphosphate (ATP), but they also have critical functions in the regulation of apoptosis.

Mitochondria release various molecules that can trigger the apoptotic pathways. One of the key players is cytochrome c, which is released from the mitochondria into the cytosol in response to pro-apoptotic signals. Once in the cytosol, cytochrome c interacts with apoptotic protease activating factor-1 (Apaf-1) to form a complex that activates caspases, the enzymes that execute the death program in the cell.

Additionally, the mitochondrial membrane potential can change during apoptosis, leading to the release of other pro-apoptotic factors, such as Smac/DIABLO and Omi, which help to inhibit the inhibitors of apoptosis and promote cell death.

Furthermore, the balance between pro-apoptotic and a